In [ ]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.scraping.scrape_reservoirs import get_reservoir_province_rate_limited

In [ ]:
water_path = PATHS['cleaned_data_notebooks']/ 'water_cleaned.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

In [ ]:
reservoirs_path = PATHS['processed_data_notebooks'] / 'post_cleaning' / 'reservoirs_cleaned.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

In [ ]:
detailed_reservoirs_path = PATHS['processed_data_notebooks'] / 'post_cleaning' / 'detailed_reservoirs_cleaned.csv'
detailed_reservoirs_pd = pd.read_csv(detailed_reservoirs_path)
detailed_reservoirs_pd.head()

### Create the merged dataframe from reservoirs, with even the non-matching entries

In [ ]:
merged_pd = pd.merge(reservoirs_pd, detailed_reservoirs_pd, on='name', how='outer')
not_paired_reservoirs = merged_pd[(merged_pd['id'].isnull()) | (merged_pd['longitude'].isnull())]['name'].copy()
not_paired_reservoirs.sort_values(inplace=True)

#### Let's display all not paired reservoirs:

In [ ]:
pd.set_option('display.max_rows', None)
not_paired_reservoirs

#### Number of already paired reservoirs:

In [ ]:
paired_reservoirs = merged_pd[(merged_pd['id'].notnull()) & (merged_pd['longitude'].notnull())]['name'].copy()
paired_reservoirs.sort_values(inplace=True)
print(f"The number of already paired reservoirs is: {len(paired_reservoirs)}")
print(f"Remember that reservoirs_pd has {len(reservoirs_pd)} rows and detailed_reservoirs_pd has {len(detailed_reservoirs_pd)} rows")

#### As detailed_reservoirs_pd has less reservoirs, we will focus on its not paired ones

In [ ]:
not_paired_from_detailed = merged_pd[merged_pd['id'].isnull()]['name'].copy()
not_paired_from_detailed.sort_values(inplace=True)
not_paired_from_detailed

In [ ]:
pd.reset_option('display.max_rows')

The correspondence between reservoirs is developed in the cleaning merges.ipynb notebook

### Looking for possible inconsistencies between dataframes

Reading the dataframe developed at the cleaning file

In [ ]:
reservoirs_path = PATHS['pre_EDA']/ 'merges_for_EDA.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

Merging them

In [ ]:
water_pd = pd.merge(water_non_merged_pd, reservoirs_pd[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

### Looking for reservoirs with higher storage than capacity

In [ ]:
water_pd[water_pd['storage'] > water_pd['capacity']]

### Show a barplot with the proportion of reservoirs with storage higher than capacity

In [ ]:
higher_count = water_pd[water_pd['storage'] > water_pd['capacity']]['id'].nunique()
valid_count = water_pd[water_pd['storage'] <= water_pd['capacity']]['id'].nunique()

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(['Higher than Capacity', 'Within Capacity'], [higher_count, valid_count], color=['orange', 'lightblue'])
plt.title('Count of Reservoirs with Storage Higher than Capacity')
plt.ylabel('Count')
plt.show()

This problem will be tackled setting the storage of the reservoirs that are over capacity to their capacity value

### Looking for reservoirs that don't have data up to date

In [ ]:
max_date = water_pd['date'].max()
print(f"The maximum date in the water dataframe is {max_date}.")

### Graphic to see the proportion of reservoirs with and without data up to date

In [ ]:
reservoirs_with_data = water_pd[water_pd['date'] == max_date]
reservoirs_without_data = reservoirs_pd[~reservoirs_pd['id'].isin(reservoirs_with_data['id'])]

plt.bar(['With Data', 'Without Data'], [reservoirs_with_data['id'].nunique(), reservoirs_without_data['id'].nunique()])
plt.title('Reservoirs Data Availability')
plt.ylabel('Number of Reservoirs')
plt.show()